<a href="https://colab.research.google.com/github/Gervais-59/Portfolio-Data-Science/blob/main/code_odyssee_rag.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Votre propre assistant IA, construit et mis en ligne ce soir

**Préparé par Machine Learnia (https://www.machinelearnia.com/)**

Bienvenue ! Dans ce notebook, nous allons construire un assistant IA capable de répondre à des questions sur un ensemble de documents que vous allez **aller chercher vous-même sur Internet**, puis nous le  mettrons **en ligne**, à une adresse **éphémaire** que vous pourrez ensuite envoyer à qui vous voulez.

(Cette adresse sera unique pour chacun d'entres vous)

**Si vous débutez, voila ce qu'il faut savoir**

- Vous exécutez chaque cellule avec **Shift + Entrée**, en même temps que moi.
- Certaines cellules ont des **trous** `______` : on les remplit ensemble, en direct. Ce sont les lignes qui comptent.
- Vous n'avez pas besoin de tout comprendre ce soir. Vous avez besoin de tout **faire**. Les explications arrivent au fur et à mesure.
- Une erreur ? Vérifiez que vous avez bien exécuté les cellules précédentes dans l'ordre, ou demandez dans le chat.

*Première étape : menu **« Fichier → Enregistrer une copie dans Drive »** pour travailler sur VOTRE copie.*

## Jalon 0 · Préparation (~2 minutes)

**Avant de lancer** : menu « Exécution » → « Modifier le type d'exécution » → choisissez **« T4 GPU »** s'il est proposé. Pas de GPU disponible ? Tout marchera quand même, un peu plus lentement.

On installe les outils du soir : le lecteur de PDF, le moteur de recherche sémantique, et l'atelier de mise en ligne.

In [1]:
%pip install -q sentence-transformers gradio pypdf
print("✅ Installation terminée !")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 385.1/385.1 kB 8.1 MB/s eta 0:00:00
✅ Installation terminée !


In [2]:
import torch

if torch.cuda.is_available():
    MODELE = "Qwen/Qwen2.5-1.5B-Instruct"
    DEVICE = 0
    print("🚀 GPU détecté : on utilise le moteur rapide.")
else:
    MODELE = "Qwen/Qwen2.5-0.5B-Instruct"
    DEVICE = -1
    print("🐢 Pas de GPU : on utilise le moteur léger. Tout marchera quand même.")

print()
print("👉 Écrivez PRÊT dans le chat !")

🚀 GPU détecté : on utilise le moteur rapide.

👉 Écrivez PRÊT dans le chat !


## Jalon 1 · Les documents

Un assistant IA d'entreprise répond à partir de **documents**. Les nôtres, ce soir : les guides pratiques du **Club Odyssée**, une franchise de salles de sport avec cinq clubs en France (Paris, Lyon, Marseille, Toulouse, Lille) et ses conditions générales. Six PDF, qu'on va **télécharger** comme on le ferait avec les documents d'un client.

Pourquoi une franchise fictive ? Parce que l'IA ne peut rien en savoir. Si votre assistant répond juste tout à l'heure sur les horaires de Lyon, vous saurez que c'est grâce à VOTRE travail, pas à sa mémoire.

In [6]:
import io, requests
from pypdf import PdfReader

# ⬇️ L'adresse publique où sont rangés les PDF (dataset Hugging Face) ⬇️
BASE_URL = "https://huggingface.co/datasets/Gui3/Atelier-Code-3-sep-25/resolve/main/"
FICHIERS = ["club-odyssee-paris.pdf", "club-odyssee-lyon.pdf", "club-odyssee-marseille.pdf",
            "club-odyssee-toulouse.pdf", "club-odyssee-lille.pdf", "club-odyssee-conditions-generales.pdf"]

TEXTES = {}
try:
    for f in FICHIERS:
        r = requests.get(BASE_URL + f, timeout=30)
        r.raise_for_status()
        pages = PdfReader(io.BytesIO(r.content)).pages
        TEXTES[f] = "\n".join((p.extract_text() or "") for p in pages)
        print(f"✅ {f:42s} {len(pages)} page(s) · {len(TEXTES[f]):5d} caractères")
except Exception as e:
    import base64
    print("⚠️ Téléchargement impossible (", e, ")")
    print("   On bascule sur les six PDF de secours embarqués : la suite est identique.\n")
    TEXTES = {}
    for f, contenu_b64 in PDF_SECOURS.items():
        pages = PdfReader(io.BytesIO(base64.b64decode(contenu_b64))).pages
        TEXTES[f] = "\n".join((p.extract_text() or "") for p in pages)
        print(f"🛟 {f:42s} {len(pages)} page(s) · {len(TEXTES[f]):5d} caractères  (secours)")

print(f"\n📚 {len(TEXTES)} documents chargés.")

✅ club-odyssee-paris.pdf                     4 page(s) · 11497 caractères
✅ club-odyssee-lyon.pdf                      4 page(s) · 10671 caractères
✅ club-odyssee-marseille.pdf                 4 page(s) · 10753 caractères
✅ club-odyssee-toulouse.pdf                  4 page(s) · 10285 caractères
✅ club-odyssee-lille.pdf                     4 page(s) · 10753 caractères
✅ club-odyssee-conditions-generales.pdf      3 page(s) ·  6957 caractères

📚 6 documents chargés.


### Le chunking : pourquoi on découpe

On ne va pas donner un document entier au modèle. Deux raisons :

1. **Sa fenêtre de contexte est finie** : on ne peut pas tout y mettre.
2. **Un passage court se retrouve mieux qu'un document entier.** Si vous cherchez les horaires de Lyon, vous voulez le paragraphe des horaires de Lyon, pas les quatre pages du guide.

Découper un texte en passages, c'est le **chunking**. Le nôtre est volontairement simple : des passages d'environ **500 caractères**, coupés de préférence en fin de phrase, avec un petit **chevauchement** pour ne pas perdre une information à cheval sur deux passages. En production, on fait plus sophistiqué. Ce soir, ça suffit largement.

Et pourquoi ne pas tout donner au modèle d'un coup ? Nos six guides font environ 25 pages, soit 15 000 tokens. Un modèle local s'y perd, c'est lent, et en entreprise on n'a pas 6 documents mais 6 000. On lui donne trois passages, les bons.

Une astuce de métier, gratuite : on **encode le titre du document avec le passage**. Ainsi le moteur sait que « ouvert de 6h30 à 22h30 » parle de Lyon, pas de Paris. En revanche on donne au modèle le passage **seul** : si on lui colle le titre dedans, il le recopie dans sa réponse. Détail minuscule, effet immédiat.

In [7]:
def decouper(texte, taille=500, chevauchement=80):
    """Découpe un texte en passages d'environ `taille` caractères, en coupant de préférence en fin de phrase."""
    texte = " ".join(texte.split())          # on nettoie les sauts de ligne du PDF
    passages, debut = [], 0
    while debut < len(texte):
        fin = min(debut + taille, len(texte))
        if fin < len(texte):
            coupe = texte.rfind(". ", debut + taille // 2, fin)   # la fin de phrase la plus proche
            if coupe != -1:
                fin = coupe + 1
        passages.append(texte[debut:fin].strip())
        if fin >= len(texte):
            break
        debut = max(fin - chevauchement, debut + 1)   # on recule un peu : le chevauchement
    return [p for p in passages if p]


DOCUMENTS = []
for f, t in TEXTES.items():
    nom = f.replace(".pdf", "").replace("club-odyssee-", "Club Odyssée ").replace("-", " ").title()
    for i, p in enumerate(decouper(t), 1):
        DOCUMENTS.append({"titre": f"{nom} · passage {i}", "texte": p})

print(f"✅ {len(DOCUMENTS)} passages, prêts à être encodés.")
print()
print("Exemple ·", DOCUMENTS[0]["titre"])
print(DOCUMENTS[0]["texte"][:350], "...")

✅ 167 passages, prêts à être encodés.

Exemple · Club Odyssée Paris · passage 1
Club Odyssée Paris Bastille Guide pratique 2026-2027 - 12 rue de la Roquette, 75011 Paris Bienvenue au club Le Club Odyssée Paris Bastille est le premier club de la franchise, ouvert en septembre 2016 dans une ancienne imprimerie de la rue de la Roquette. Réparti sur deux niveaux et 1 800 m², il accueille aujourd'hui environ 2 300 abonnés, avec une ...


## Jalon 2 · Le moteur qui comprend le sens

Comment retrouver le bon passage quand quelqu'un pose une question ? Pas avec des mots-clés : avec des **embeddings**.

Le principe : chaque passage est transformé en un vecteur, une liste de nombres qui capture son **sens**. La question subit le même sort. Il ne reste qu'à comparer les vecteurs : les plus proches sont les passages qui parlent de la même chose.

Le modèle qu'on utilise pour ça, vous l'avez déjà vu mardi : c'est **celui qui a cartographié les 2 117 offres d'emploi**.

In [9]:
from sentence_transformers import SentenceTransformer
import numpy as np

# TROU 1 · le modèle de mardi
encodeur = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

# On encode le TITRE avec le passage : le moteur saura ainsi de quel club on parle.
# Mais le texte qu'on donnera au modèle, lui, restera propre.()
textes = [f"{d['titre']} {d['texte']}" for d in DOCUMENTS]
vecteurs = encodeur.encode(textes, normalize_embeddings=True)

print(f"✅ {len(vecteurs)} passages encodés.")
print(f"Chaque passage est devenu un vecteur de {vecteurs.shape[1]} nombres.")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

✅ 167 passages encodés.
Chaque passage est devenu un vecteur de 384 nombres.


In [10]:
# TROU 2 · combien de passages on donne au modèle ?
def chercher(question, k=3):
    """Renvoie les k passages les plus proches de la question."""
    v_question = encodeur.encode(question, normalize_embeddings=True)
    similarites = vecteurs @ v_question
    indices = np.argsort(-similarites)[:k]
    return [(DOCUMENTS[i]["titre"], DOCUMENTS[i]["texte"], float(similarites[i]))
            for i in indices]

for titre, texte, score in chercher("Le club de Lyon a-t-il une piscine ?"):
    print(f"[{score:.2f}] {titre}")

print()
print("✅ Le moteur retrouve les bons passages, et le bon club !")

[0.78] Club Odyssée Lyon · passage 26
[0.72] Club Odyssée Lyon · passage 6
[0.70] Club Odyssée Lyon · passage 7

✅ Le moteur retrouve les bons passages, et le bon club !


### La preuve que c'est du sens, pas des mots

Posons une question dont **aucun mot** n'apparaît dans le bon passage : « Je veux arrêter mon abonnement ». Le passage qui répond parle de « résiliation », jamais d'« arrêter ». Regardez.

In [11]:
for titre, texte, score in chercher("Je veux arrêter mon abonnement, comment faire ?"):
    print(f"[{score:.2f}] {titre}")

[0.49] Club Odyssée Lille · passage 11
[0.48] Club Odyssée Paris · passage 11
[0.47] Club Odyssée Marseille · passage 9


## Jalon 3 · L'IA qui rédige... et d'abord, une expérience

Il nous manque la troisième brique : le modèle de langage qui va rédiger les réponses.

On va le charger, puis faire une expérience en deux temps:

1. On va lui poser une question **sans** lui donner le moindre document.
2. **Ensuite** on lui posera exactement la même, avec les documents. Et on affichera les deux réponses l'une sous l'autre.

Choisissez une question dont la réponse est **vérifiable** dans nos guides : un prix, un horaire, un équipement. Vous saurez ainsi, sans discussion possible, si la réponse est juste ou non.

In [12]:
from transformers import pipeline
import transformers

transformers.logging.set_verbosity_error()   # on masque les avertissements techniques

print("Chargement du modèle (1 à 3 minutes la première fois)...")
generateur = pipeline("text-generation", model=MODELE, device=DEVICE)

# On règle la génération une fois pour toutes
generateur.tokenizer.clean_up_tokenization_spaces =False #
generateur.model.generation_config.max_new_tokens = 200
generateur.model.generation_config.do_sample = False
generateur.model.generation_config.temperature = False # pour faire varier les reponse
generateur.model.generation_config.top_p = None
generateur.model.generation_config.top_k = None


print("✅ Modèle chargé !")



Chargement du modèle (1 à 3 minutes la première fois)...


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 3.09GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/7.30k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/7.03M [00:00<?, ?B/s]

✅ Modèle chargé !


In [13]:
def demander_au_modele(messages):
    sortie = generateur(messages)
    return sortie[0]["generated_text"][-1]["content"]

In [14]:
# Dans toute vraie application, l'assistant a un RÔLE. Le nôtre :
ROLE = ("Tu es l'assistant du Club Odyssée, une chaîne de salles de sport en France. "
        "Tu réponds aux questions des adhérents de façon précise, utile et directe.")

In [22]:
# votre question : sa réponse est dans nos guides, mais le modèle ne les a pas encore.
question_test = "Combien coûte l'abonnement premieur au club odysée à Marseille?"   # ex : Combien coûte l'abonnement Premium au club de Marseille ?

# On garde cette réponse de côté : on la comparera tout à l'heure.
reponse_sans_documents = demander_au_modele([
    {"role": "system", "content": ROLE},          # il est l'assistant du club...
    {"role": "user", "content": question_test},   # ...mais il n'a aucun document.
])
print(reponse_sans_documents)

Désolé, mais je n'ai pas d'informations spécifiques sur les tarifs de l'abonnement premier pour le Club Odyssée à Marseille. Ces détails peuvent varier selon la région et l'année. Pour obtenir les informations exactes, il serait préférable de contacter le service client du club ou de consulter leur site web le plus récemment mis à jour. Ils seront mieux placés pour fournir les informations les plus fiables et actualisées concernant les tarifs actuels.


### Que vient-il de se passer ?

Le Club Odyssée **n'existe pas**. Le modèle n'avait donc aucun moyen de connaître la réponse. Et pourtant il a produit quelque chose.

Regardez votre écran, puis le chat : **deux choses différentes viennent de se produire dans la salle.**

- Chez certains, il a **inventé** une réponse. Un prix, un horaire, avec assurance. C'est une **hallucination** : quand un modèle ne sait pas, il complète, parce que compléter est tout ce qu'il sait faire.
- Chez d'autres, il a **refusé** : « je n'ai pas cette information, contactez le club ».


Une seule cause aux deux pannes : **il n'a pas vos documents.**

Regardez bien la cellule suivante : **le rôle ne change pas**. Ce sont les documents et la règle qui s'ajoutent.

In [23]:
def repondre(question):
    """L'assistant complet : recherche + rédaction sous contrainte."""
    passages = chercher(question) # on recherches les K passages les plus pertinents vis-a-vis de la question
    contexte = "\n\n".join(f"## {t}\n{x}" for t,x,_,in passages) # on mets tous ces "passages" au sein d'une meme string, ca sera notre "contexte"

    # On Construit le Prompt Final : qu'est-ce qu'on lui interdit ? et que doit-il dire s'il ne sait pas ?
    messages = [
        {"role": "system", "content": ROLE +          # <- le MÊME rôle que tout à l'heure
            " Tu réponds ______ à partir des documents fournis. "
            "Si la réponse n'y figure pas, réponds exactement : « Les informations dont je dispose me permettent pas de répondre à cette question »"},
        {"role": "user", "content": f"Documents :\n{contexte}\n\nQuestion : {question}"},
    ]

    reponse = demander_au_modele(messages)
    sources = ", ".join(t for t, _, _ in passages)
    return reponse, sources


# La MÊME question, cette fois avec les documents. On affiche les deux réponses l'une sous l'autre.
reponse_avec_documents, sources = repondre(question_test)

print("❓", question_test)
print()
print("❌ SANS les documents :")
print("   ", " ".join(reponse_sans_documents.split())[:400])
print()
print("✅ AVEC les documents :")
print("   ", " ".join(reponse_avec_documents.split()))
print("    📎 Sources :", sources)

❓ Combien coûte l'abonnement premieur au club odysée à Marseille?

❌ SANS les documents :
    Désolé, mais je n'ai pas d'informations spécifiques sur les tarifs de l'abonnement premier pour le Club Odyssée à Marseille. Ces détails peuvent varier selon la région et l'année. Pour obtenir les informations exactes, il serait préférable de contacter le service client du club ou de consulter leur site web le plus récemment mis à jour. Ils seront mieux placés pour fournir les informations les plu

✅ AVEC les documents :
    L'abonnement Premium au club Odysée à Marseille coûte 42,90 euros par mois.
    📎 Sources : Club Odyssée Marseille · passage 7, Club Odyssée Marseille · passage 8, Club Odyssée Marseille · passage 3


In [24]:
questions = [
    "Le club de Lyon a-t-il une piscine ?",
    "Quel club propose un sauna ?",
    "Le club de Toulouse est-il ouvert le dimanche ?",
    "Combien de cours collectifs par semaine propose le club de Lyon ?",
    "Quel est le préavis pour résilier mon abonnement ?",
    "Quel est le cours du Bitcoin ?",
]

for q in questions:
    r, s = repondre(q)
    print(f"❓ {q}")
    print(f"💬 {r}")
    print(f"📎 {s}")
    print()

print("🎉 Levez la main dans le chat : votre assistant vient de répondre !")

❓ Le club de Lyon a-t-il une piscine ?
💬 Non.
📎 Club Odyssée Lyon · passage 26, Club Odyssée Lyon · passage 6, Club Odyssée Lyon · passage 7

❓ Quel club propose un sauna ?
💬 Le Club Odyssée Lille propose un sauna.
📎 Club Odyssée Lille · passage 26, Club Odyssée Lille · passage 6, Club Odyssée Lille · passage 27

❓ Le club de Toulouse est-il ouvert le dimanche ?
💬 Non.
📎 Club Odyssée Toulouse · passage 2, Club Odyssée Toulouse · passage 5, Club Odyssée Toulouse · passage 25

❓ Combien de cours collectifs par semaine propose le club de Lyon ?
💬 Le club de Lyon propose 38 cours collectifs par semaine.
📎 Club Odyssée Lyon · passage 3, Club Odyssée Lyon · passage 13, Club Odyssée Lyon · passage 25

❓ Quel est le préavis pour résilier mon abonnement ?
💬 Selon la condition générale du Club Odyssée, le préavis pour résilier son abonnement est de deux mois. Cela signifie que si vous souhaitez rompre votre abonnement avant la date anniversaire, vous devez le faire au moins deux mois plus tôt.
📎

Notez la dernière question : le Bitcoin n'est pas dans les documents, et l'assistant **le dit** au lieu d'inventer. C'est exactement la différence entre utiliser l'IA et la maîtriser.

Et remarquez ce qui a changé entre les deux réponses de tout à l'heure : **pas le modèle**, il est identique. **Pas le rôle**, il est identique. Ce sont les trois passages et une règle de huit mots. C'est tout le RAG.

## Jalon 4 · LA MISE EN LIGNE

Votre assistant fonctionne. Mais il vit dans ce notebook, sur cette page. Personne d'autre que vous ne peut s'en servir.

On va lui donner deux choses : un **visage** (une vraie page web) et une **adresse publique**. Un seul mot crée un tunnel : votre application, qui tourne ici, devient accessible depuis n'importe où dans le monde.

In [25]:
import gradio as gr

def assistant_web(question):
    if not question.strip():
        return "Posez-moi une question sur les clubs Odyssée !"
    reponse, sources = repondre(question)
    return f"{reponse}\n\n📎 Sources : {sources}"

demo = gr.Interface(
    fn=assistant_web,
    inputs=gr.Textbox(label="Votre question",
                      placeholder="Ex : le club de Lyon a-t-il une piscine ?"),
    outputs=gr.Textbox(label="Réponse de l'assistant"),
    title="Mon premier assistant IA 🤖",
    description="Il répond aux questions sur les clubs Odyssée, une franchise de salles de sport fictive "
                "(Paris, Lyon, Marseille, Toulouse, Lille). Construit en direct avec Machine Learnia.",
)

# TROU 5 · un seul mot pour passer de votre écran au monde
demo.launch(share=True)

Colab notebook detected. To show errors in colab notebook, set debug=True in launch()
* Running on public URL: https://728d194253450a8e04.gradio.live

This share link is temporary and will last for up to 1 week (best effort). For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)


### 📱 Le moment

Dans la sortie ci-dessus, cherchez la ligne :

> `Running on public URL: https://xxxxxxxx.gradio.live`

Maintenant, dans l'ordre :

1. **Ouvrez ce lien dans un autre navigateur, ou sur votre téléphone.**
2. **Posez une question** à votre assistant.
3. **Envoyez le lien à quelqu'un. Maintenant.** Votre conjoint, un ami. Avec le message de votre choix. Le nôtre serait : « regarde ce que je viens de construire ».

Puis revenez nous dire dans le chat ce que ça fait. 😊

**À savoir** : ce lien est temporaire, il vivra **quelques heures** puis s'éteindra. Donc si vous voulez le partager avec d'autres personnes (par exemple sur linkedin, NE PARTAGER PAS CET URL... demain il sera désactivé)

## Le bilan

Ce soir, vous avez construit les quatre briques de tout assistant IA d'entreprise :

1. **Des documents** allés chercher sur Internet, puis **découpés en passages** (le chunking).
2. **Un moteur de recherche sémantique**, qui comprend le sens et pas seulement les mots.
3. **Un modèle qui rédige sous contrainte** : il répond à partir de vos documents, cite ses sources, et dit « je ne sais pas » plutôt que d'inventer.
4. **Une mise en ligne**, à une adresse publique.

Ce qui est temporaire, c'est le lien. Ce qui est à vous, c'est le savoir-faire, et ce notebook.

**Pour les bâtisseurs du Cahier de Vacances** : votre application du Projet 7 mérite le même traitement. La méthode change un peu (Streamlit a son propre service de mise en ligne, gratuit lui aussi), le principe est identique.

À samedi ! ✌️

**Guillaume - Machine Learnia**